# Langfuse로 RAG 파이프라인 모니터링하기

RAG 체인이 실행될 때 검색 결과, 프롬프트, LLM 응답, 지연 시간, 비용을 Langfuse에 기록하고 분석하는 방법을 실습합니다.

필요한 것: `.env`에 `OPENAI_API_KEY`, `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_HOST`

## 0. 실습 준비: 벡터 DB 만들기

아래 실습은 `data/신입사원민수.pdf`로 만든 FAISS 벡터 DB(`faiss_index_minsu`)를 사용합니다. 폴더가 없으면 새로 만듭니다(임베딩 API 비용이 조금 듭니다).

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

DB_PATH = "./faiss_index_minsu"
if not os.path.isdir(DB_PATH):
    docs = PyPDFLoader("data/신입사원민수.pdf").load()
    chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100).split_documents(docs)
    FAISS.from_documents(chunks, OpenAIEmbeddings(model="text-embedding-3-small")).save_local(DB_PATH)
    print("벡터 DB 생성:", len(chunks), "청크")
else:
    print("기존 벡터 DB 사용:", DB_PATH)

## 7. RAG 운영 및 모니터링 (Monitoring & Observability)

### 7.1 왜 RAG에는 모니터링이 필요한가?

RAG 시스템은 단순한 LLM 호출이 아니라  
**검색(Retrieval)과 생성(Generation)이 결합된 파이프라인**입니다.

따라서 답변 품질이 저하되었을 때, 그 원인은 여러 단계 중 하나일 수 있습니다.

- 검색된 문서가 질문과 잘 맞지 않았는가?
- 검색 문서는 적절했지만 프롬프트 구성이 문제였는가?
- context 길이가 과도하게 길었는가?
- LLM 생성 과정에서 의미 왜곡이 발생했는가?

이러한 문제는 **최종 답변만 보고는 판단하기 어렵습니다.**  
RAG 시스템에서는 단순한 결과 평가를 넘어,  **과정 전체를 관찰할 수 있는 모니터링(Observability)** 이 필요합니다.

<img src="image/RAG.png" width="600">


### 7.2 RAG Observability 개념과 도구 선택

Observability란 시스템의 내부 상태를 외부에서 관찰 가능한 정보만으로 추론할 수 있는 능력을 의미합니다.

RAG 관점에서 Observability는 다음과 같은 질문에 답할 수 있어야 합니다.

- 어떤 질문이 입력되었는가?
- Retriever는 어떤 문서들을 반환했는가?
- 실제 LLM에 전달된 context는 무엇이었는가?
- 응답 생성까지 어느 단계에서 시간이 소요되었는가?

이를 위해 RAG 시스템에서는 다음과 같은 도구들이 활용됩니다.

- LangSmith
- Langfuse

이러한 도구들은 모두 **RAG 요청 단위의 Trace를 수집하고 분석**하는 것을 목표로 합니다.

### 7.3 LangSmith vs Langfuse 비교

LangSmith와 Langfuse는 목적은 유사하지만, 성격과 활용 환경에는 차이가 있습니다.

### 공통점
- RAG 요청 단위 추적
- Prompt / Context / Output 기록
- 성능 분석을 위한 로그 제공

### 차이점 개요

| 구분 | LangSmith | Langfuse |
|-----|----------|----------|
| 개발 주체 | LangChain | Langfuse |
| 프레임워크 의존성 | LangChain 중심 | 프레임워크 독립 |
| 배포 방식 | 관리형 중심 | 오픈소스 / 자체 호스팅 가능 |
| 기업 환경 적합성 | 빠른 실험용 | 내부 시스템 연동에 유리 |

Langfuse는 **오픈소스 기반**이며, 자체 서버에 설치하여 운영할 수 있기 때문에  
기업 환경에서 RAG 시스템을 운영할 때 자주 선택됩니다.

본 강의에서는 **실무 환경에서 활용하기 적합한 RAG 모니터링 도구의 예시**로  
Langfuse를 사용합니다.

### 7.4 Langfuse 시작하기

### 1) Langfuse 회원가입 및 조직(Organization) 생성
- https://langfuse.com 접속
- 회원가입 진행
- 로그인 후 Organization 생성  
  (Langfuse는 모든 프로젝트가 조직 단위로 관리됨)
  
  <img src="image/langfuse_organization.png" width=650>  
  
  <br>
  
  <img src="image/langfuse_organization2.png" width=650>

### 2) 프로젝트(Project) 생성  

- 생성한 Organization 내부에서 새 프로젝트 생성  
- 프로젝트 이름 및 기본 설정 입력  

  <img src="image/langfuse_project.png" width=650>

### 3) API Key 발급
- Project Settings 메뉴 이동  

  <img src="image/langfuse_api_key.png" width=650>

- Public Key / Secret Key 확인  

  <img src="image/langfuse_api_key2.png" width=650>

### 4) .env 파일에 환경 변수 설정

```bash
LANGFUSE_PUBLIC_KEY=...
LANGFUSE_SECRET_KEY=...
LANGFUSE_HOST=https://cloud.langfuse.com
```

이후 Python 코드에서 load_dotenv 함수를 통해 환경 변수를 로드하여 사용합니다.

```python
from dotenv import load_dotenv
load_dotenv(override=True)
```

본 장에서는 위에서 이미 구축한 벡터 DB와 `dense_retriever`를 이용하여,  
 **RAG 체인 실행 결과를 Langfuse에 기록하는 방법**에 집중합니다.


### 7.5 실습: Langfuse CallbackHandler로 RAG 요청 추적하기

Langfuse의 `CallbackHandler`는  
**체인 실행 시점에 전달하여 전체 실행 흐름을 추적**하는 방식으로 사용합니다.

이 방식을 사용할 경우,

- Retriever 검색 단계
- Context 구성
- Prompt 적용
- LLM 호출
- 최종 Output 생성

까지 **RAG 파이프라인 전체가 하나의 Trace로 Langfuse에 기록**됩니다.

#### 1) Langfuse CallbackHandler 생성

In [1]:
from langfuse.langchain import CallbackHandler
langfuse_handler = CallbackHandler()

#### 2) RAG 체인 실행 (CallbackHandler 전달)

이미 구성된 LCEL 기반 RAG 체인을 실행할 때,  
`config` 인자를 통해 CallbackHandler를 전달합니다.

In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS

# 문서 리스트를 하나의 문자열 context로 변환
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG 프롬프트
rag_prompt = ChatPromptTemplate.from_template(
    """
    다음 문서를 참고하여 질문에 답변하세요.
    문서에 없는 내용은 "없는 정보"라고 답변하세요.

    [문서]
    {context}

    [질문]
    {question}
    """
)

# LLM 설정
llm = ChatOpenAI(
    model="gpt-4o-mini",
    
    temperature=0
)

# FAISS 벡터 스토어 로드
DB_PATH = "./faiss_index_minsu"
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.load_local(
    folder_path=DB_PATH,
    embeddings=embeddings,
    allow_dangerous_deserialization=True  # 필수 설정
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# LCEL 기반 RAG 체인 구성
rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriever | format_docs,
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# 질의 실행 (Langfuse CallbackHandler 연결)
question = "신입사원 민수가 세 번째 부서에서 만난 개발자는 왜 쉬지 않고 일했나요?"

answer = rag_chain.invoke(
    question,
    config={
        "callbacks": [langfuse_handler]
    }
)

print(answer)


신입사원 민수가 세 번째 부서에서 만난 개발자는 "부끄럽다는 걸 잊기 위해서" 코딩을 하고 있다고 말했습니다.


이 방식으로 실행하면, 
Retriever → Prompt → LLM → Output 전체 흐름이 Langfuse에 자동으로 저장됩니다.

<img src="image/langfuse_tracing.png" width="800">

### 7.6 Langfuse 주요 기능 살펴보기

Langfuse는 단순히 하나의 RAG 요청 결과만 확인하는 도구가 아니라, RAG 시스템의 실행 흐름을 기록하고, 결과 품질을 평가하며,  
프롬프트와 데이터셋을 기반으로 실험과 개선을 반복할 수 있도록 다양한 관측(Observability) 기능을 제공합니다.

아래 표는 Langfuse에서 제공하는 주요 기능과 각 기능의 역할을 정리한 것입니다.

| 영역                    | 기능               | 설명                                          | 활용 목적            |
| --------------------- | ---------------- | ------------------------------------------- | ---------------- |
| **Tracing**           | Trace            | 하나의 사용자 요청(RAG 요청) 단위 실행 흐름                 | 요청 단위 디버깅, 성능 비교 |
|                       | Observation      | Trace 내부의 개별 실행 단계 (LLM 호출, Retriever 실행 등) | 단계별 병목 분석        |
| **Sessions / Users**  | Sessions         | 여러 요청을 하나의 세션으로 묶어 관리                       | 사용자 흐름 분석        |
|                       | Users            | 사용자 단위 실행 기록 관리                             | 사용자별 사용 패턴 분석    |
| **Prompt Management** | Prompts          | 프롬프트 템플릿을 자산으로 관리                           | 프롬프트 버전 관리, 실험   |
| **Evaluation**        | Scores           | 출력 결과에 점수 부여                                | 품질 정량 평가         |
|                       | LLM-as-a-Judge   | 다른 LLM을 평가자로 활용                             | 자동화된 대규모 평가      |
|                       | Human Annotation | 사람이 직접 결과를 평가                               | 정성 평가, 기준 검증     |
|                       | Datasets         | 질문·정답 기준 데이터셋 관리                            | 반복 실험, 성능 비교     |
| **Dashboards**        | Dashboards       | 주요 지표를 요약해 시각화                              | 운영 상태 모니터링       |


<img src="image/langfuse_main.png" width="800">


### 7.7 Langfuse를 활용한 품질 분석 사례

Langfuse에 기록된 로그를 통해 다음과 같은 분석이 가능합니다.

- 답변이 부정확한 요청에서
  - Retriever 결과가 적절했는지
  - Context가 과도하게 길지는 않았는지
- 동일한 질문에 대해
  - 서로 다른 프롬프트 버전이 어떤 영향을 주는지
- 응답 품질 저하가
  - 검색 단계 문제인지
  - 생성 단계 문제인지

이는 **정량 평가 점수 없이도**  
문제의 원인을 구조적으로 추적할 수 있게 해줍니다.

### 7.8 평가(Evaluation)와 모니터링의 역할 차이

Evaluation과 Monitoring은 서로 다른 목적을 가집니다.

- Evaluation
  - 결과가 좋은지 판단
  - 정답률, 점수 중심
- Monitoring
  - 왜 이런 결과가 나왔는지 분석
  - 과정과 맥락 중심

실무 환경에서는 두 접근 방식이 함께 사용됩니다.

- Evaluation은 기준을 만들고
- Monitoring은 개선 방향을 찾는 역할을 합니다.

### 7.9 정리

- RAG 시스템은 한 번 구축하고 끝나는 시스템이 아닙니다.
- 운영 과정에서 지속적인 관찰과 분석이 필요합니다.
- Langfuse와 같은 Observability 도구를 활용하면
  - RAG 내부 동작을 이해할 수 있고
  - 성능 문제를 체계적으로 개선할 수 있습니다.